# Packet P-040 — Prediction Error Predictability (Error Meta-Model)

**Decision question:** Can we predict which devices the model will get wrong, using data-quality proxies? If yes, we can flag unreliable predictions at inference time.

**Fastest falsifier:** Logistic regression on meta-features predicting above-median error. If ROC-AUC < 0.55, error is not predictable.

**Success:** Meta-model ROC-AUC ≥ 0.65.
**Kill:** ROC-AUC < 0.55 — errors are random, not predictable from available signals.

In [1]:
"""Cell 1 — Collect residuals and build meta-features."""
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('perovskite_stability_clean.csv')
FEATURES = [
    'Perovskite_band_gap', 'Pb', 'Sn', 'I', 'Br', 'Cl',
    'MA', 'FA', 'Cs',
    'first_Prvskt_annealing_temperature', 'first_Prvskt_thermal_annealing_time',
    'Perovskite_thickness', 'Cell_area_measured',
    'JV_default_Voc', 'JV_default_Jsc', 'JV_default_FF'
]
TARGET = 'Stability_PCE_T80'
ET_PARAMS = dict(n_estimators=700, max_features='sqrt', min_samples_split=3,
                 min_samples_leaf=1, bootstrap=False, random_state=42)

X = df[FEATURES].fillna(0)
y = np.log1p(df[TARGET])

# Collect residuals across 10 splits
N_SPLITS = 10
all_residuals = {}  # idx -> list of abs errors

for seed in range(N_SPLITS):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    
    model = ExtraTreesRegressor(**ET_PARAMS)
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    
    # Per-tree std for uncertainty signal
    tree_preds = np.array([t.predict(X_te) for t in model.estimators_])
    tree_std = tree_preds.std(axis=0)
    
    for i, (idx, pred, actual) in enumerate(zip(X_te.index, preds, y_te)):
        if idx not in all_residuals:
            all_residuals[idx] = {'abs_errors': [], 'tree_stds': []}
        all_residuals[idx]['abs_errors'].append(abs(pred - actual))
        all_residuals[idx]['tree_stds'].append(tree_std[i])

    if (seed + 1) % 5 == 0:
        print(f"  Completed {seed+1}/{N_SPLITS} splits")

# Build meta-feature matrix
def classify_family(row):
    ma, fa, cs = row["MA"] > 0, row["FA"] > 0, row["Cs"] > 0
    if ma and not fa and not cs: return "Pure MA"
    elif fa and not ma and not cs: return "Pure FA"
    elif ma and fa and not cs: return "MA-FA mixed"
    elif fa and cs and not ma: return "FA-Cs"
    elif ma and fa and cs: return "Triple cation"
    else: return "Other"

df["family"] = df.apply(classify_family, axis=1)

# Meta-features (NOT model input features — to avoid circularity)
meta_rows = []
for idx, res in all_residuals.items():
    row = df.loc[idx]
    mean_err = np.mean(res['abs_errors'])
    mean_tree_std = np.mean(res['tree_stds'])
    
    # Missingness count
    n_missing = row[FEATURES].isna().sum()
    
    # Ref_ID quartile (temporal proxy)
    ref_q = pd.qcut(df['Ref_ID'], q=4, labels=False).loc[idx] if 'Ref_ID' in df.columns else 0
    
    # Number of extreme features (|z| > 2)
    z_scores = (X.loc[idx] - X.mean()) / (X.std() + 1e-8)
    n_extreme = (np.abs(z_scores) > 2).sum()
    
    # JV features missing
    jv_missing = row[['JV_default_Jsc', 'JV_default_Voc', 'JV_default_FF']].isna().sum()
    
    meta_rows.append({
        'idx': idx,
        'mean_abs_error': mean_err,
        'mean_tree_std': mean_tree_std,
        'n_missing': n_missing,
        'ref_quartile': ref_q,
        'n_extreme': n_extreme,
        'jv_missing': jv_missing,
        'family': row['family'],
    })

meta_df = pd.DataFrame(meta_rows)
print(f"\nMeta-features built for {len(meta_df)} devices")
print(f"Mean abs error: {meta_df['mean_abs_error'].mean():.4f}")
print(f"Median abs error: {meta_df['mean_abs_error'].median():.4f}")

  Completed 5/10 splits


  Completed 10/10 splits



Meta-features built for 1374 devices
Mean abs error: 1.2565
Median abs error: 0.9821


In [2]:
"""Cell 2 — Train meta-model and evaluate."""

# Binary target: above-median error
median_err = meta_df['mean_abs_error'].median()
meta_df['high_error'] = (meta_df['mean_abs_error'] > median_err).astype(int)

# Meta-feature matrix
family_dummies = pd.get_dummies(meta_df['family'], prefix='fam')
meta_features = pd.concat([
    meta_df[['mean_tree_std', 'n_missing', 'ref_quartile', 'n_extreme', 'jv_missing']],
    family_dummies
], axis=1).values

meta_target = meta_df['high_error'].values

# Scale
scaler = StandardScaler()
meta_features_scaled = scaler.fit_transform(meta_features)

# 5-fold cross-validated AUC
aucs = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    meta_features_scaled, meta_target,
    cv=5, scoring='roc_auc'
)

print(f"{'=' * 60}")
print("ERROR META-MODEL RESULTS")
print(f"{'=' * 60}")
print(f"Meta-model: Logistic Regression")
print(f"Meta-features: tree_std, n_missing, ref_quartile, n_extreme, jv_missing, family dummies")
print(f"5-fold CV ROC-AUC: {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}")
print(f"Per-fold AUCs: {[f'{a:.3f}' for a in aucs]}")

# Feature importance (coefficients)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(meta_features_scaled, meta_target)

feature_names = ['tree_std', 'n_missing', 'ref_quartile', 'n_extreme', 'jv_missing'] + list(family_dummies.columns)
coefs = pd.Series(lr.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)

print(f"\n── Top meta-feature coefficients (predicting high error) ──")
for feat, coef in coefs.head(8).items():
    print(f"  {feat:<30} {coef:+.4f}")

# Status
mean_auc = np.mean(aucs)
if mean_auc >= 0.65:
    status = "Confirmed"
elif mean_auc >= 0.55:
    status = "Promising"
else:
    status = "Negative"

# Save
save_df = pd.DataFrame({'feature': feature_names, 'coefficient': lr.coef_[0]})
save_df.loc[len(save_df)] = {'feature': 'mean_auc', 'coefficient': mean_auc}
save_df.to_csv('Packet_P040_Error_Meta_Model.csv', index=False)
print(f"\nSaved: Packet_P040_Error_Meta_Model.csv")

print(f"\n{'=' * 60}")
print(f"P-040 STATUS: {status}")
print(f"{'=' * 60}")
if status == "Confirmed":
    print("Error is predictable from meta-features — can flag unreliable predictions.")
    print(f"Top predictor: {coefs.index[0]} (coef {coefs.iloc[0]:+.4f})")
elif status == "Promising":
    print("Weak but non-trivial error predictability.")
else:
    print("Errors are random — not predictable from available signals.")

ERROR META-MODEL RESULTS
Meta-model: Logistic Regression
Meta-features: tree_std, n_missing, ref_quartile, n_extreme, jv_missing, family dummies
5-fold CV ROC-AUC: 0.5960 +/- 0.0125
Per-fold AUCs: ['0.601', '0.611', '0.591', '0.602', '0.574']

── Top meta-feature coefficients (predicting high error) ──
  tree_std                       +0.4291
  jv_missing                     -0.1020
  n_missing                      +0.0921
  fam_Triple cation              -0.0718
  n_extreme                      +0.0692
  fam_Pure MA                    +0.0311
  fam_MA-FA mixed                +0.0290
  fam_FA-Cs                      -0.0106

Saved: Packet_P040_Error_Meta_Model.csv

P-040 STATUS: Promising
Weak but non-trivial error predictability.
